# Day 5: SQLite, Full-Stack Architecture & Capstone Project

Welcome to the final day of **Learn Python in 5 Days**.

## What You Will Learn Today
- Relational database fundamentals & the zero-configuration **SQLite** engine
- Executing safe **parameterized SQL queries** (`?` placeholders) with Python's built-in `sqlite3` module
- Using **`conn.row_factory = sqlite3.Row`** for dictionary-like column access
- Wiring real SQLite persistence into the **FastAPI Service Layer**
- Writing automated test suites using **`pytest`** and **`TestClient`**
- Comprehensive review of the 5-day journey and launching your **Capstone Project**

---

## 1. Connecting to SQLite & Creating Tables
Python includes the `sqlite3` module in its standard library. SQLite stores your entire relational database inside a single `.db` file on disk.

In [ ]:
import sqlite3
from pathlib import Path

# 1. Create an in-memory SQLite database for interactive testing
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row  # Access columns by name

cursor = conn.cursor()

# 2. Create tasks table
cursor.execute("""
CREATE TABLE tasks (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    priority TEXT DEFAULT 'Normal',
    completed INTEGER DEFAULT 0,
    created_at TEXT NOT NULL
);
""")
conn.commit()
print("Tasks table created successfully in SQLite!")

## 2. Parameterized Queries (Preventing SQL Injection)
> **CRITICAL SECURITY RULE:** Never format raw user strings into SQL queries using f-strings. Always pass variables via the **`?`** placeholder tuple.

In [ ]:
from datetime import datetime, timezone

# 1. Safe Parameterized Insertion
new_tasks = [
    ("Implement SQLite persistence", "High", 1, datetime.now(timezone.utc).isoformat()),
    ("Write pytest integration test", "High", 0, datetime.now(timezone.utc).isoformat()),
    ("Launch capstone project", "Urgent", 0, datetime.now(timezone.utc).isoformat())
]

cursor.executemany("""
INSERT INTO tasks (title, priority, completed, created_at)
VALUES (?, ?, ?, ?);
""", new_tasks)
conn.commit()

print(f"Inserted {cursor.rowcount} tasks safely using parameterized queries.")

## 3. Querying, Updating & Deleting Records
With `conn.row_factory = sqlite3.Row`, rows act like dictionaries: `row["title"]`.

In [ ]:
# 1. Query all tasks
cursor.execute("SELECT * FROM tasks WHERE completed = 0 ORDER BY id DESC;")
pending_tasks = cursor.fetchall()

print("--- Pending Tasks in SQLite ---")
for row in pending_tasks:
    print(f"[#{row['id']}] {row['title']:<32} | Priority: {row['priority']:<6} | Done: {bool(row['completed'])}")

# 2. Update task
cursor.execute("UPDATE tasks SET completed = 1 WHERE id = ?;", (2,))
conn.commit()

# 3. Verify update
cursor.execute("SELECT * FROM tasks WHERE id = ?;", (2,))
updated_row = cursor.fetchone()
print(f"\nUpdated Task #2 completed status:", bool(updated_row["completed"]))

## 4. Testing the Full FastAPI + SQLite Application
Let's import our complete Day 5 FastAPI application and verify the entire request pipeline using `TestClient`.

In [ ]:
from fastapi.testclient import TestClient
from app.main import app

client = TestClient(app)

# 1. Test GET /tasks
res = client.get("/tasks")
print("GET /tasks Status:", res.status_code)
print(f"Fetched {len(res.json())} persistent tasks from SQLite:")
for t in res.json():
    print(f" - #{t['id']}: {t['title']} ({t['priority']})")

## 5. Automated Testing with `pytest` Patterns
In Python, writing automated tests is as simple as defining functions named `test_*()` with standard `assert` statements.

In [ ]:
def test_create_and_retrieve_task_lifecycle():
    # 1. Create a new task via API POST
    payload = {
        "title": "End-to-End Pytest Verification",
        "priority": "High",
        "tags": ["qa", "pytest"]
    }
    post_res = client.post("/tasks", json=payload)
    assert post_res.status_code == 201, "Should return 201 Created"
    created_data = post_res.json()
    task_id = created_data["id"]
    assert created_data["title"] == payload["title"]

    # 2. Query the created task via GET
    get_res = client.get(f"/tasks/{task_id}")
    assert get_res.status_code == 200, "Should return 200 OK"
    assert get_res.json()["id"] == task_id

    # 3. Delete task
    del_res = client.delete(f"/tasks/{task_id}")
    assert del_res.status_code == 204, "Should return 204 No Content"

    # 4. Verify 404
    missing_res = client.get(f"/tasks/{task_id}")
    assert missing_res.status_code == 404, "Should return 404 Not Found"
    
    print("All test assertions passed successfully!")

# Run test function directly
test_create_and_retrieve_task_lifecycle()

---
## 6. Course Graduation & Capstone Project Selection

### Suggested Capstone Options:
1. **Expense Tracker & Financial Dashboard:** FastAPI + Pydantic + SQLite aggregations + React/Chart.js frontend.
2. **Automated Bookmark Manager:** HTTPX + BeautifulSoup4 metadata harvester + SQLite store.
3. **Website & API Uptime Monitor:** Async ping scheduler + response latency tracking.
4. **GitHub Developer Analytics Dashboard:** Public API consumer + SQLite cache.

### Minimum Capstone Deliverables:
- `pyproject.toml` managed with `uv`
- Decoupled 3-layer architecture (`models.py`, `services.py`, `main.py`)
- SQLite persistence with parameterized queries
- Working user interface (Web UI or CLI)
- Minimum 3 automated test cases with `pytest`